# ARGUS enrollment embeddings (T4 GPU)

Generates ArcFace embeddings for the full enrollment gallery (LFW 5,749 identities + MFR2 53 identities, one unmasked photo each) on Colab's free T4 - the same job takes ~53 min on the local CPU, a few minutes here.

**Before running:** upload `enrollment_images.zip` and `enrollment_manifest.csv` (both produced locally by `enrollment/package_for_colab.py` and `enrollment/build_manifest.py`) to a folder in your Google Drive, e.g. `MyDrive/argus_enrollment/`. Update `DRIVE_DIR` below to match.

In [ ]:
!pip install -q insightface onnxruntime-gpu opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/argus_enrollment'

In [ ]:
import zipfile
import os

EXTRACT_DIR = '/content/enrollment_images'
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(os.path.join(DRIVE_DIR, 'enrollment_images.zip')) as zf:
    zf.extractall(EXTRACT_DIR)

print(len(os.listdir(EXTRACT_DIR)), 'images extracted')

In [ ]:
from insightface.app import FaceAnalysis

# det_size=160, not the InsightFace default 640 - LFW/MFR2 raw images are small
# (250x250 / 160x160), and 640 upsamples them enough to break SCRFD's anchor
# matching (same issue we hit and fixed locally in embeddings/generate.py)
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider'])
app.prepare(ctx_id=0, det_size=(160, 160))

In [ ]:
import csv

with open(os.path.join(DRIVE_DIR, 'enrollment_manifest.csv'), newline='') as f:
    manifest_rows = list(csv.DictReader(f))

# package_for_colab.py renamed each file to dataset_identity_filename to keep the zip flat
def archive_name(row):
    return f"{row['dataset']}_{row['identity']}_{row['filename']}"

print(len(manifest_rows), 'rows in manifest')

In [ ]:
import cv2
import numpy as np

def pick_largest_face(faces):
    if not faces:
        return None
    return max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))

kept_rows, embeddings = [], []
skipped = 0

for i, row in enumerate(manifest_rows):
    img_path = os.path.join(EXTRACT_DIR, archive_name(row))
    image = cv2.imread(img_path)
    if image is None:
        skipped += 1
        continue
    face = pick_largest_face(app.get(image))
    if face is None:
        skipped += 1
        continue
    kept_rows.append(row)
    embeddings.append(face.normed_embedding.astype(np.float32))
    if (i + 1) % 500 == 0:
        print(f'{i + 1}/{len(manifest_rows)} processed, {skipped} skipped')

print(f'done: {len(kept_rows)} embeddings, {skipped} skipped')

In [ ]:
out_path = os.path.join(DRIVE_DIR, 'enrollment_embeddings.npz')
np.savez_compressed(
    out_path,
    dataset=np.array([r['dataset'] for r in kept_rows]),
    identity=np.array([r['identity'] for r in kept_rows]),
    filename=np.array([r['filename'] for r in kept_rows]),
    embedding=np.stack(embeddings),
)
print('saved to', out_path)

`enrollment_embeddings.npz` is now in your Drive folder. Download it (or sync the Drive folder locally) and run `enrollment/seed_chromadb.py` against it on your machine.